# Setup

In [1]:
pip install pandas numpy transformers huggingface matplotlib scikit-learn seaborn torch tqdm

Note: you may need to restart the kernel to use updated packages.


# Finetune

Paradigm:
1) Raw text is fed into the word embedding model
2) Word embedding concatenates with feature engineered numerical values
3) Classification is done for the mental health status label
4) Loss is computed and gradient descent is done on the model
5) Repeat until the model converges to a solution

In [2]:
# Grab BERT model
from transformers import DistilBertTokenizer, DistilBertModel
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOKENIZER = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
MODEL = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Move to device
print(DEVICE)
MODEL = MODEL.to(DEVICE)

# Unfreeze parameters
for param in MODEL.parameters():
    param.requires_grad = True

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cuda


In [3]:
# Grab dataset
import pandas as pd 

train_df = pd.read_csv('features_train.csv')
val_df = pd.read_csv('features_val.csv')
test_df = pd.read_csv('features_test.csv')

In [4]:
# Exclude the 'normal' class since it is an outlier

train_df = train_df[train_df['status'] != 'normal']
val_df = val_df[val_df['status'] != 'normal']
test_df = test_df[test_df['status'] != 'normal']

In [5]:
# Initialize some columns for finetuning

# print(train_df.columns)
text_column = 'statement'
label_column = 'status'

In [6]:
# Prepare output labels
from sklearn.preprocessing import LabelEncoder
import torch

# Encode labels into numerical values
label_encoder = LabelEncoder()
train_df[label_column] = label_encoder.fit_transform(train_df[label_column])
val_df[label_column] = label_encoder.transform(val_df[label_column])

# Turn into tensors
train_df[label_column] = [torch.tensor(label, dtype=torch.long) for label in train_df[label_column]]
val_df[label_column] = [torch.tensor(label, dtype=torch.long) for label in val_df[label_column]]

In [7]:
# Prepare input statements
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 16

# Define a custom class for getting dataset attributes
class CustomDataset(Dataset):
    def __init__(self, df, text_column, label_column):
        self.df = df
        self.text_column = text_column
        self.label_column = label_column
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return {
            'text': self.df.iloc[idx][self.text_column],
            'label': self.df.iloc[idx][self.label_column],
        }

# Set custom class
train_data = CustomDataset(train_df, text_column, label_column)
val_data = CustomDataset(val_df, text_column, label_column)

# Create dataloaders for batching
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

In [8]:
# Prepare the model for finetuning
import torch.nn as nn

# Initialize vector sizes 
hidden_size = 768
numerical_size = len(['first_person', 'negatives', 'suicide', 'reps'])
concatenated_size = hidden_size + numerical_size
num_classes = int(train_df[label_column].max()) + 1 # Due to 0 indexing

# Initialize a simple neural network
dropout = 0.5 # Following what Laasya did
CLASSIFIER = nn.Sequential(
    nn.Linear(concatenated_size, hidden_size), # layer 1
    nn.Dropout(dropout), 
    nn.ReLU(),
    nn.Linear(hidden_size, num_classes),
)
CLASSIFIER = CLASSIFIER.to(DEVICE)

# Initialize optimizer and loss function
optimizer = torch.optim.AdamW(list(MODEL.parameters()) + list(CLASSIFIER.parameters()), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [9]:
# Define a forward pass

# Defined in make_features_helper.py
def word_ratio(text: str, provided_words):
  """
  Takes in a single piece of text and counts for the ratio of words in that text that are provided_words
  """
  words = text.split()
  count = 0
  for word in words:
    if word in provided_words:
      count += 1
  return count/max(len(words), 1)
    
# Defined in make_features_helper.py
def reps_ratio(text: str):
  """
  Takes in a single piece of text and counts how many words are repeated
  """
  words = text.split()
  unique_words = set()
  for word in words:
    unique_words.add(word)
  return 1 - len(unique_words) / max(len(words), 1)
    
# For word ratios
FIRST_PERSON = ['i', 'me', 'my', 'mine', 'myself']
NEGATIVES = ['no', 'not', 'never', 'nothing', 'wrong', 'nope']
SUICIDE_WORDS = ['die', 'end', 'forever', 'leave', 'gone', 'suicide', 'kill']

def forward(texts):

    # Get encoding
    encoding = TOKENIZER(texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
    encoding = {k: v.to(DEVICE) for k, v in encoding.items()}
    outputs = MODEL(**encoding)
    embeddings = outputs.last_hidden_state[:, 0, :]
    embeddings = embeddings.to(DEVICE)

    # Get numerical features
    numerical_features = []
    for text in texts:
        first_person_ratio = word_ratio(text, FIRST_PERSON)
        negatives_ratio = word_ratio(text, NEGATIVES)
        suicide_ratio = word_ratio(text, SUICIDE_WORDS)
        repeat_ratio = reps_ratio(text)
        numerical_features.append([first_person_ratio, negatives_ratio, suicide_ratio, repeat_ratio])
    numerical_features = torch.tensor(numerical_features, dtype=torch.float32)
    numerical_features = numerical_features.to(DEVICE)
        
    # Concatenate
    combined_features = torch.cat([embeddings, numerical_features], dim=1)
    combined_features = combined_features.to(DEVICE)
    
    # Classification
    logits = CLASSIFIER(combined_features)
    
    # Make sure RAM doesn't blow up
    del embeddings
    del outputs
    del numerical_features
    del combined_features
    torch.cuda.empty_cache()

    return logits


In [10]:
# Main finetuning loop

print("Training...")
print()

num_epochs = 10

for epoch in range(num_epochs):

    CLASSIFIER.train()
    MODEL.train()
    train_loss = 0

    for batch in train_loader:

        # Move to device
        texts = batch['text']
        labels = batch['label'].to(DEVICE)

        # Forward prop
        optimizer.zero_grad()
        logits = forward(texts)
        loss = criterion(logits, labels)

        # Backward prop
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Cleanup
        del labels
        torch.cuda.empty_cache()
        
        train_loss += loss.item()
        

    # Validate
    CLASSIFIER.eval()
    MODEL.eval()
    val_loss = 0
    correct_preds = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:

            # Move to device
            texts = batch['text']
            labels = batch['label'].to(DEVICE)

            # Forward 
            logits = forward(texts)
            loss = criterion(logits, labels)

            # Calculate 
            val_loss += loss.item()
            y_pred = torch.argmax(logits, dim=1)
            correct_preds += (y_pred == labels).sum().item()
            total += labels.size()[0]

            # Cleanup
            del labels
            torch.cuda.empty_cache()

    # Print metrics
    print(f"Epoch {epoch+1} done")
    print(f" Train loss: {train_loss/len(train_loader)}")
    print(f" Val loss: {val_loss/len(val_loader)}")
    print(f" Val accuracy: {100*correct_preds/total}%")
    print()

    # Save
    torch.save({
        'bert_state_dict': MODEL.state_dict(),
        'classifier_state_dict': CLASSIFIER.state_dict(),
    }, f"model_epoch{epoch+1}.pt")
    
print("Done")

Training...

Epoch 1 done
 Train loss: 0.7450411561899964
 Val loss: 0.5850188660918793
 Val accuracy: 75.10548523206751%

Epoch 2 done
 Train loss: 0.48999366434583874
 Val loss: 0.5465080195333252
 Val accuracy: 78.13245276096129%

Epoch 3 done
 Train loss: 0.3670289822508517
 Val loss: 0.5848860927909351
 Val accuracy: 77.56374977068428%

Epoch 4 done
 Train loss: 0.26323316338114766
 Val loss: 0.6239224684225324
 Val accuracy: 77.91230966795084%

Epoch 5 done
 Train loss: 0.1823415923810933
 Val loss: 0.688848303756581
 Val accuracy: 77.94900018345258%

Epoch 6 done
 Train loss: 0.11619828811703167
 Val loss: 0.8140177523766445
 Val accuracy: 77.30691616217207%

Epoch 7 done
 Train loss: 0.08078848761359944
 Val loss: 0.9892582127119789
 Val accuracy: 76.04109337736195%

Epoch 8 done
 Train loss: 0.06117330683437733
 Val loss: 1.0573088906465038
 Val accuracy: 76.42634379013025%

Epoch 9 done
 Train loss: 0.051027827294698604
 Val loss: 1.0135760633548028
 Val accuracy: 75.41735461

# Hyperparameter tuning

I consolidated everything in the above cells into 1 cell for easier hyperparameter tuning. For proper documentation and comments, please refer to the above cells instead.

In [8]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
import torch

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
DROPOUT = 0.3
NUM_EPOCHS = 20
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1

# 1) Grab the dataset & preprocess the data
print("Loading data...")
train_df = pd.read_csv('features_train.csv')
val_df = pd.read_csv('features_val.csv')
test_df = pd.read_csv('features_test.csv')
train_df = train_df[train_df['status'] != 'normal']
val_df = val_df[val_df['status'] != 'normal']
test_df = test_df[test_df['status'] != 'normal']
text_column = 'statement'
label_column = 'status'
label_encoder = LabelEncoder()
train_df[label_column] = label_encoder.fit_transform(train_df[label_column])
val_df[label_column] = label_encoder.transform(val_df[label_column])
test_df[label_column] = label_encoder.transform(test_df[label_column])
train_df[label_column] = [torch.tensor(label, dtype=torch.long) for label in train_df[label_column]]
val_df[label_column] = [torch.tensor(label, dtype=torch.long) for label in val_df[label_column]]
test_df[label_column] = [torch.tensor(label, dtype=torch.long) for label in test_df[label_column]]

# 2) Format data into dataloaders
print("Formatting data...")
class CustomDataset(Dataset):
    def __init__(self, df, text_column, label_column):
        self.df = df
        self.text_column = text_column
        self.label_column = label_column
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return {
            'text': self.df.iloc[idx][self.text_column],
            'label': self.df.iloc[idx][self.label_column],
        }
train_data = CustomDataset(train_df, text_column, label_column)
val_data = CustomDataset(val_df, text_column, label_column)
test_data = CustomDataset(test_df, text_column, label_column)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_df, batch_size=BATCH_SIZE)

# 3) Set up the pretrained model + fully connected model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOKENIZER = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
MODEL = DistilBertModel.from_pretrained('distilbert-base-uncased')
print("Using:", DEVICE)
MODEL = MODEL.to(DEVICE)
for param in MODEL.parameters():
    param.requires_grad = False
hidden_size = 768
numerical_size = len(['first_person', 'negatives', 'suicide', 'reps'])
concatenated_size = hidden_size + numerical_size
num_classes = int(train_df[label_column].max()) + 1
CLASSIFIER = nn.Sequential(
    nn.Linear(concatenated_size, hidden_size),
    nn.Dropout(DROPOUT), 
    nn.ReLU(),
    nn.Linear(hidden_size, num_classes),
)
CLASSIFIER = CLASSIFIER.to(DEVICE)
def word_ratio(text: str, provided_words):
  words = text.split()
  count = 0
  for word in words:
    if word in provided_words:
      count += 1
  return count/max(len(words), 1)
def reps_ratio(text: str):
  words = text.split()
  unique_words = set()
  for word in words:
    unique_words.add(word)
  return 1 - len(unique_words) / max(len(words), 1)
FIRST_PERSON = ['i', 'me', 'my', 'mine', 'myself']
NEGATIVES = ['no', 'not', 'never', 'nothing', 'wrong', 'nope']
SUICIDE_WORDS = ['die', 'end', 'forever', 'leave', 'gone', 'suicide', 'kill']
def forward(texts):
    encoding = TOKENIZER(texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
    encoding = {k: v.to(DEVICE) for k, v in encoding.items()}
    outputs = MODEL(**encoding)
    embeddings = outputs.last_hidden_state[:, 0, :]
    embeddings = embeddings.to(DEVICE)
    numerical_features = []
    for text in texts:
        first_person_ratio = word_ratio(text, FIRST_PERSON)
        negatives_ratio = word_ratio(text, NEGATIVES)
        suicide_ratio = word_ratio(text, SUICIDE_WORDS)
        repeat_ratio = reps_ratio(text)
        numerical_features.append([first_person_ratio, negatives_ratio, suicide_ratio, repeat_ratio])
    numerical_features = torch.tensor(numerical_features, dtype=torch.float32)
    numerical_features = numerical_features.to(DEVICE)
    combined_features = torch.cat([embeddings, numerical_features], dim=1)
    combined_features = combined_features.to(DEVICE)
    logits = CLASSIFIER(combined_features)
    del embeddings
    del outputs
    del numerical_features
    del combined_features
    torch.cuda.empty_cache()
    return logits

# 4) Set up training for these models
optimizer = torch.optim.AdamW([{'params': MODEL.parameters(), 'lr': 2e-5}, {'params': CLASSIFIER.parameters(), 'lr': 1e-4}], weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("Training...")
print()
for epoch in range(NUM_EPOCHS):
    CLASSIFIER.train()
    if epoch == (NUM_EPOCHS // 2):
        for param in MODEL.parameters():
            param.requires_grad = True
            MODEL.train()
    train_loss = 0
    for batch in train_loader:
        texts = batch['text']
        labels = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = forward(texts)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        del labels
        torch.cuda.empty_cache()
        train_loss += loss.item()
    CLASSIFIER.eval()
    MODEL.eval()
    val_loss = 0
    correct_preds = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            texts = batch['text']
            labels = batch['label'].to(DEVICE)
            logits = forward(texts)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            y_pred = torch.argmax(logits, dim=1)
            correct_preds += (y_pred == labels).sum().item()
            total += labels.size()[0]
            del labels
            torch.cuda.empty_cache()
    print(f"Epoch {epoch+1} done")
    print(f" Train loss: {train_loss/len(train_loader)}")
    print(f" Val loss: {val_loss/len(val_loader)}")
    print(f" Val accuracy: {100*correct_preds/total}%")
    print()
    torch.save({
        'bert_state_dict': MODEL.state_dict(),
        'classifier_state_dict': CLASSIFIER.state_dict(),
    }, f"model_epoch{epoch+1}.pt")
print("Done")


Loading data...
Formatting data...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using: cuda
Training...

Epoch 1 done
 Train loss: 1.2186665650433715
 Val loss: 1.1246778318259723
 Val accuracy: 61.21812511465786%

Epoch 2 done
 Train loss: 1.090859734187336
 Val loss: 1.0710078927079254
 Val accuracy: 64.17171161254815%

Epoch 3 done
 Train loss: 1.051404645907804
 Val loss: 1.045989017857135
 Val accuracy: 66.40983305815446%

Epoch 4 done
 Train loss: 1.0290791066187732
 Val loss: 1.0359902168648683
 Val accuracy: 65.620986974867%

Epoch 5 done
 Train loss: 1.0129471919446622
 Val loss: 1.015427490658075
 Val accuracy: 67.49220326545588%

Epoch 6 done
 Train loss: 0.9993730551416768
 Val loss: 1.0062346830745597
 Val accuracy: 67.51054852320675%

Epoch 7 done
 Train loss: 0.9911912327667453
 Val loss: 0.9957830000483046
 Val accuracy: 68.84975233902037%

Epoch 8 done
 Train loss: 0.9806325632821089
 Val loss: 0.9933323758788123
 Val accuracy: 69.03320491652907%

Epoch 9 done
 Train loss: 0.9719473593639878
 Val loss: 0.9808346092526165
 Val accuracy: 69.85874151

Run 1:
- BATCH_SIZE = 16
- LEARNING_RATE = 2e-5
- DROPOUT = 0.3
- NUM_EPOCHS = 10
- WEIGHT_DECAY = 0.1

Validation Accuracy:
- Epoch 1: 75.00%
- Epoch 2: 78.04%
- Epoch 3: 77.44%
- Epoch 4: 77.31%
- Epoch 5: 77.64%
- Epoch 6: 77.84%
- Epoch 7: 77.33%
- Epoch 8: 77.22%
- Epoch 9: 77.34%
- Epoch 10: 77.27%


Run 2:
- 30 Epochs
- Same hyperparameters otherwise
- Froze BERT weights for half the epochs

Results:

Epoch 1 done
 Train loss: 1.2486315547670208
 Val loss: 1.095458210038999
 Val accuracy: 53.641533663547975%

Epoch 2 done
 Train loss: 1.0464326104652957
 Val loss: 1.0083167886803925
 Val accuracy: 58.79655109154284%

Epoch 3 done
 Train loss: 0.9800222952785732
 Val loss: 0.9591000025922601
 Val accuracy: 60.96129150614566%

Epoch 4 done
 Train loss: 0.9368038877001349
 Val loss: 0.9238570573742438
 Val accuracy: 61.36488717666483%

Epoch 5 done
 Train loss: 0.9076216094523856
 Val loss: 0.8974795292549469
 Val accuracy: 63.16272243625023%

Epoch 6 done
 Train loss: 0.8835978186542883
 Val loss: 0.8789722627558666
 Val accuracy: 64.04329480829206%

Epoch 7 done
 Train loss: 0.8651504428694083
 Val loss: 0.8592654089948648
 Val accuracy: 64.63034305631994%

Epoch 8 done
 Train loss: 0.8478400855881613
 Val loss: 0.8433011638279185
 Val accuracy: 65.05228398458998%

Epoch 9 done
 Train loss: 0.8338181813183071
 Val loss: 0.8311098015203504
 Val accuracy: 65.41918913960741%

Epoch 10 done
 Train loss: 0.8219454859412691
 Val loss: 0.8219978049004183
 Val accuracy: 65.69436800587049%

Epoch 11 done
 Train loss: 0.809415571550903
 Val loss: 0.8135091283104636
 Val accuracy: 66.17134470739313%

Epoch 12 done
 Train loss: 0.800451168399187
 Val loss: 0.8083574570868372
 Val accuracy: 65.84113006787746%

Epoch 13 done
 Train loss: 0.792654273513728
 Val loss: 0.7991134484142851
 Val accuracy: 66.3914878004036%

Epoch 14 done
 Train loss: 0.7864763216215109
 Val loss: 0.7918891435669314
 Val accuracy: 67.03357182168409%

Epoch 15 done
 Train loss: 0.7784952469404388
 Val loss: 0.7860362028970747
 Val accuracy: 67.34544120344891%

Epoch 16 done
 Train loss: 0.6091839287828349
 Val loss: 0.5509155632114131
 Val accuracy: 75.76591451109888%

Epoch 17 done
 Train loss: 0.42223872512860117
 Val loss: 0.5131424364368936
 Val accuracy: 77.87561915244909%

Epoch 18 done
 Train loss: 0.2377120299756808
 Val loss: 0.6379117498562134
 Val accuracy: 77.45367822417904%

Epoch 19 done
 Train loss: 0.09875282028407362
 Val loss: 0.769442326730647
 Val accuracy: 77.83892863694734%

Epoch 20 done
 Train loss: 0.04766302096916278
 Val loss: 1.0129234479341513
 Val accuracy: 77.87561915244909%

Epoch 21 done
 Train loss: 0.0364928260078628
 Val loss: 1.1258543138050876
 Val accuracy: 77.63713080168776%

Epoch 22 done
 Train loss: 0.03241506017185437
 Val loss: 1.1463478693497555
 Val accuracy: 76.463034305632%

Epoch 23 done
 Train loss: 0.027554735119804906
 Val loss: 1.0501851723024136
 Val accuracy: 76.86662997615116%

Epoch 24 done
 Train loss: 0.023116696302692943
 Val loss: 1.3328962597416993
 Val accuracy: 76.8482847184003%

Epoch 25 done
 Train loss: 0.024228059123405664
 Val loss: 1.2845203424633216
 Val accuracy: 77.74720234819299%

Epoch 26 done
 Train loss: 0.019060223487909644
 Val loss: 1.2695220220189465
 Val accuracy: 77.12346358466337%

Epoch 27 done
 Train loss: 0.01869916536411009
 Val loss: 1.2100652939014551
 Val accuracy: 76.44468904788113%

Epoch 28 done
 Train loss: 0.01987408771518366
 Val loss: 1.4070651036498747
 Val accuracy: 77.08677306916162%

Epoch 29 done
 Train loss: 0.01615251290211462
 Val loss: 1.4251329654682963
 Val accuracy: 76.9767015226564%

Epoch 30 done
 Train loss: 0.020615094798137973
 Val loss: 1.3544643857732108
 Val accuracy: 76.15116492386719%

Notes:
- Overfitting by Epoch 17 -> 18. We can cap it at 20 epochs.


Run 3: 
- 20 Epochs
- Use a smaller LR for BERT, larger LR for classifier
- Same hyperparameters as above otherwise

Results:

Epoch 1 done
 Train loss: 1.0595802851068148
 Val loss: 0.9245571127035751
 Val accuracy: 61.18143459915612%

Epoch 2 done
 Train loss: 0.8840384223925992
 Val loss: 0.8452833462670402
 Val accuracy: 64.92386718033389%

Epoch 3 done
 Train loss: 0.8268840318393408
 Val loss: 0.8118170992719812
 Val accuracy: 65.58429645936525%

Epoch 4 done
 Train loss: 0.7954365053829157
 Val loss: 0.7918805241409984
 Val accuracy: 66.61163089341406%

Epoch 5 done
 Train loss: 0.7747922855915513
 Val loss: 0.7855645391074094
 Val accuracy: 66.1346541918914%

Epoch 6 done
 Train loss: 0.7557214786609013
 Val loss: 0.7565889043891885
 Val accuracy: 68.00587048248028%

Epoch 7 done
 Train loss: 0.7433072432219607
 Val loss: 0.7450361917794974
 Val accuracy: 68.13428728673638%

Epoch 8 done
 Train loss: 0.73156727979768
 Val loss: 0.7411474961339554
 Val accuracy: 68.5011924417538%

Epoch 9 done
 Train loss: 0.7199157866948056
 Val loss: 0.7358463843547004
 Val accuracy: 68.77637130801688%

Epoch 10 done
 Train loss: 0.7118625448184943
 Val loss: 0.7336240972766429
 Val accuracy: 68.48284718400294%

Epoch 11 done
 Train loss: 0.5985920354161622
 Val loss: 0.5230505073393894
 Val accuracy: 77.27022564667034%

Epoch 12 done
 Train loss: 0.4184021154828604
 Val loss: 0.5058587976127775
 Val accuracy: 78.71950100898917%

Epoch 13 done
 Train loss: 0.2528927675325353
 Val loss: 0.5524168339642611
 Val accuracy: 78.11410750321042%

Epoch 14 done
 Train loss: 0.1149621787131398
 Val loss: 0.9403988323788927
 Val accuracy: 76.29792698587416%

Epoch 15 done
 Train loss: 0.05162227355167431
 Val loss: 1.1788962115233224
 Val accuracy: 76.79324894514768%

Epoch 16 done
 Train loss: 0.044226846630857
 Val loss: 1.1152564408530719
 Val accuracy: 77.50871399743167%

Epoch 17 done
 Train loss: 0.03486545897221388
 Val loss: 1.0169339213094228
 Val accuracy: 76.95835626490552%

Epoch 18 done
 Train loss: 0.029355785651809265
 Val loss: 1.4528267498830023
 Val accuracy: 76.64648688314071%

Epoch 19 done
 Train loss: 0.028634605502276325
 Val loss: 1.3006825698843456
 Val accuracy: 76.11447440836544%

Epoch 20 done
 Train loss: 0.026440447454868253
 Val loss: 1.3710888099019918
 Val accuracy: 76.9033204916529%

Notes:
- Same solution, just longer time to get there.

Run 4:
- Label smoothing: 0.1
- Weight Decay: 0.01
- Same hyperparameters otherwise

Results:

Epoch 1 done
 Train loss: 1.2186665650433715
 Val loss: 1.1246778318259723
 Val accuracy: 61.21812511465786%

Epoch 2 done
 Train loss: 1.090859734187336
 Val loss: 1.0710078927079254
 Val accuracy: 64.17171161254815%

Epoch 3 done
 Train loss: 1.051404645907804
 Val loss: 1.045989017857135
 Val accuracy: 66.40983305815446%

Epoch 4 done
 Train loss: 1.0290791066187732
 Val loss: 1.0359902168648683
 Val accuracy: 65.620986974867%

Epoch 5 done
 Train loss: 1.0129471919446622
 Val loss: 1.015427490658075
 Val accuracy: 67.49220326545588%

Epoch 6 done
 Train loss: 0.9993730551416768
 Val loss: 1.0062346830745597
 Val accuracy: 67.51054852320675%

Epoch 7 done
 Train loss: 0.9911912327667453
 Val loss: 0.9957830000483046
 Val accuracy: 68.84975233902037%

Epoch 8 done
 Train loss: 0.9806325632821089
 Val loss: 0.9933323758788123
 Val accuracy: 69.03320491652907%

Epoch 9 done
 Train loss: 0.9719473593639878
 Val loss: 0.9808346092526165
 Val accuracy: 69.85874151531829%

Epoch 10 done
 Train loss: 0.9653409248627957
 Val loss: 0.9828236550529681
 Val accuracy: 68.66629976151165%

Epoch 11 done
 Train loss: 0.8936826924482981
 Val loss: 0.8416044023379552
 Val accuracy: 76.73821317189507%

Epoch 12 done
 Train loss: 0.7549181600786605
 Val loss: 0.8345658520449641
 Val accuracy: 78.22417904971564%

Epoch 13 done
 Train loss: 0.6161126446049169
 Val loss: 0.8588645578829075
 Val accuracy: 77.80223812144561%

Epoch 14 done
 Train loss: 0.4957460063433497
 Val loss: 0.9404716842160546
 Val accuracy: 76.95835626490552%

Epoch 15 done
 Train loss: 0.4514720412545234
 Val loss: 0.9421767139714484
 Val accuracy: 77.56374977068428%

Epoch 16 done
 Train loss: 0.45072136508218896
 Val loss: 1.0202410060289668
 Val accuracy: 75.9493670886076%

Epoch 17 done
 Train loss: 0.4478229786802388
 Val loss: 0.9747611451183945
 Val accuracy: 77.83892863694734%

Epoch 18 done
 Train loss: 0.44708718237636974
 Val loss: 0.9688971970787496
 Val accuracy: 77.85727389469822%

Epoch 19 done
 Train loss: 0.44304824765748196
 Val loss: 1.008768653589959
 Val accuracy: 76.92166574940379%

Epoch 20 done
 Train loss: 0.4409908085874042
 Val loss: 0.9857008015829797
 Val accuracy: 77.65547605943864%

In [2]:
best_accuracy = 0
best_epoch = 0

for epoch in range(1, 11):
    checkpoint = torch.load(f"model_epoch{epoch}.pt")
    MODEL.load_state_dict(checkpoint['bert_state_dict'])
    CLASSIFIER.load_state_dict(checkpoint['classifier_state_dict'])
    
    MODEL.eval()
    CLASSIFIER.eval()
    
    correct_preds = 0
    total = 0
    
    with torch.no_grad():
        for batch in val_loader:
            texts = batch['text']
            labels = batch['label'].to(DEVICE)
            logits = forward(texts)
            y_pred = torch.argmax(logits, dim=1)
            correct_preds += (y_pred == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct_preds / total
    print(f"Epoch {epoch}: {accuracy:.2f}%")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_epoch = epoch

print(f"\nBest model: Epoch {best_epoch} with {best_accuracy:.2f}% accuracy")

# Load and save the best model
best_checkpoint = torch.load(f"model_epoch{best_epoch}.pt")
torch.save(best_checkpoint, "best_model.pt")

Epoch 1: 75.00%
Epoch 2: 78.04%
Epoch 3: 77.44%
Epoch 4: 77.31%
Epoch 5: 77.64%
Epoch 6: 77.84%
Epoch 7: 77.33%
Epoch 8: 77.22%
Epoch 9: 77.34%
Epoch 10: 77.27%

Best model: Epoch 2 with 78.04% accuracy
